# DisProt dataset exploration

Executed notebook for opening `data/processed/disprot/disprot_sequence_disorder.parquet` and checking the sequence/disorder mask dataset.

In [1]:
from pathlib import Path
from collections import Counter

import pyarrow.parquet as pq

DATASET_PATH = Path("data/processed/disprot/disprot_sequence_disorder.parquet")
print(DATASET_PATH)

data/processed/disprot/disprot_sequence_disorder.parquet


In [2]:
table = pq.read_table(DATASET_PATH)
print("rows:", table.num_rows)
print("columns:", table.column_names)

rows: 3337
columns: ['Uniprot_ID', 'organism', 'taxonomy_id', 'sequence', 'disorder_mask']


In [3]:
ids = table["Uniprot_ID"].to_pylist()
organisms = table["organism"].to_pylist()
taxonomy_ids = table["taxonomy_id"].to_pylist()
sequences = table["sequence"].to_pylist()
masks = table["disorder_mask"].to_pylist()

sequence_lengths = [len(seq) for seq in sequences]
mask_lengths = [len(mask) for mask in masks]
disorder_percents = [mask.count("1") / len(mask) * 100 if mask else 0 for mask in masks]

bad_lengths = sum(1 for seq_len, mask_len in zip(sequence_lengths, mask_lengths) if seq_len != mask_len)
print("bad sequence/mask lengths:", bad_lengths)
print("sequence length min/max/avg:", min(sequence_lengths), max(sequence_lengths), f"{sum(sequence_lengths) / len(sequence_lengths):.2f}")
print("disorder percent min/max/avg:", f"{min(disorder_percents):.2f}", f"{max(disorder_percents):.2f}", f"{sum(disorder_percents) / len(disorder_percents):.2f}")

bad sequence/mask lengths: 0
sequence length min/max/avg: 0 34350 581.90
disorder percent min/max/avg: 0.00 100.00 25.35


In [4]:
top_indices = sorted(range(len(disorder_percents)), key=lambda idx: disorder_percents[idx], reverse=True)[:5]
for idx in top_indices:
    print(ids[idx], organisms[idx], taxonomy_ids[idx], sequence_lengths[idx], f"{disorder_percents[idx]:.2f}", sep="\t")

A0A078IB52	Brassica napus	3708	150	100.00
A0A0H2UWN8	Streptococcus pyogenes serotype M3 (strain ATCC BAA-595 / MGAS315)	198466	60	100.00
A0A0M4HM24	Triticum turgidum subsp. durum	4567	212	100.00
A0A1B0GTR3	Homo sapiens	9606	108	100.00
A0A1Z3GD05	Hordeum vulgare	4513	138	100.00


In [5]:
for organism, count in Counter(organisms).most_common(5):
    print(count, organism, sep="\t")

1339	Homo sapiens
223	Saccharomyces cerevisiae (strain ATCC 204508 / S288c)
209	Mus musculus
146	Escherichia coli (strain K12)
123	Arabidopsis thaliana
